# Replica del metodo di Toma, Piltan e Kim (2021)Riproduzione del framework DAE + CNN descritto in *A Deep Autoencoder-Based ConvolutionNeural Network Framework for Bearing Fault Classification in Induction Motors*,Sensors 21, 8453, applicato ai dati reali del KAt-DataCenter dell'Universita di Paderborn.L'obiettivo e la fedelta: stessi cuscinetti della Tabella 2, stessa architettura delleTabelle 3 e 4, stesso protocollo della Sezione 4. Dove il paper non specifica unparametro la scelta e dichiarata esplicitamente nella cella corrispondente.Il notebook gira su Colab: i dati stanno su Drive, il codice viene da GitHub, irisultati (figure e tabelle) tornano su Drive.

In [ ]:
import os, sys, glob, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    su_colab = True
except ImportError:
    su_colab = False

cartelle_codice = ['.', '../codice',
                   '/content/drive/MyDrive/MacchineEdAzionamentiExam/codice',
                   '/content/MacchineEdAzionamentiExam/codice']

percorso_codice = None
for c in cartelle_codice:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break

if percorso_codice is None:
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/matpaol/MacchineEdAzionamentiExam',
                    '/content/MacchineEdAzionamentiExam'], check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam/codice'

sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='02_replica_paper')
dev = f.dispositivo()

print('codice da', percorso_codice)
print('risultati in', P['risultati'])
print('dispositivo', dev)

## 1. Cuscinetti e condizione operativaI 17 cuscinetti sono quelli della Tabella 2 del paper: sei sani, cinque con guastosulla pista esterna, sei sulla pista interna. Sono tutti a danno reale, prodotto daprove di vita accelerate, non a danno artificiale.Il paper dice di avere lavorato su "tre diverse condizioni" ma non indica quali. Ilnumero di segmenti che dichiara, 1320, permette di restringere il campo: il datasetmette a disposizione quattro regimi, e in ciascuno i 17 cuscinetti hanno 20registrazioni da 4 s, cioe 1360 segmenti da un secondo. Con due o piu regimi siarriverebbe ad almeno 2720. Quindi 1320 e compatibile soltanto con una singolacondizione operativa, ed e ragionevole leggere le "tre condizioni" come le tre*classi* di cuscinetto. Qui si usa `N15_M07_F10` (1500 rpm, 0,7 Nm, 1000 N), lacondizione nominale del banco.

In [ ]:
cuscinetti = config.CUSCINETTI_PAPER
regime = config.REGIME_PRINCIPALE

print(len(cuscinetti), 'cuscinetti,', regime, config.REGIMI[regime])
for classe, elenco in config.CUSCINETTI_PER_CLASSE.items():
    print(' ', config.NOMI_CLASSI[classe], '->', len(elenco), ':', ' '.join(elenco))

## 2. Scarico ed estrazione degli archiviGli archivi `.rar` (uno per cuscinetto, 150-180 MB ciascuno) restano su Drive: siscaricano una volta sola e vengono estratti sul disco locale della macchina Colab,che e piu veloce di Drive in lettura e viene comunque azzerato a fine sessione.

In [ ]:
f.scarica_archivi(cuscinetti, P['raw'])
f.estrai_misure(cuscinetti, P['raw'], P['estratti'])
f.estrai_schede(cuscinetti, P['raw'], P['documenti'])

## 3. Inventario, prima di qualunque calcoloUn'estrazione parziale non genera errori: genera silenziosamente meno dati. Prima dicostruire i segmenti conviene quindi censire quello che c'e davvero, su tutti equattro i regimi, e confrontarlo con quello che ci si aspetta.Il conteggio atteso e 17 cuscinetti x 20 registrazioni x 4 regimi = 1360.

In [ ]:
inv_completo = f.inventario(cuscinetti, P['estratti'])
riepilogo = f.riepilogo_inventario(inv_completo)

print(riepilogo.to_string(index=False))
print()
print('registrazioni totali:', len(inv_completo), '| attese:', 17 * 20 * 4)

per_cuscinetto = inv_completo.groupby(['cuscinetto', 'regime']).size().unstack()
incompleti = per_cuscinetto[(per_cuscinetto != 20).any(axis=1)]
if len(incompleti):
    print()
    print('cuscinetti con un numero di registrazioni diverso da 20:')
    print(incompleti.to_string())
else:
    print('tutti i cuscinetti hanno 20 registrazioni in ciascuno dei quattro regimi')

## 4. Segmenti da un secondoOgni registrazione da 4 s produce quattro segmenti non sovrapposti da 64 000 campioni.Le registrazioni leggermente piu corte di 256 000 campioni danno un segmento in meno:sono poche e vengono contate, non scartate a monte.

In [ ]:
inv = inv_completo[inv_completo['regime'] == regime].reset_index(drop=True)
segmenti, anagrafica = f.costruisci_segmenti(inv, segmenti_per_registrazione=4)

print(len(segmenti), 'segmenti da', segmenti.shape[1], 'campioni')
print('memoria:', round(segmenti.nbytes / 1e6), 'MB')
print()

conteggi = anagrafica.groupby('classe').agg(segmenti=('segmento', 'size'),
                                            registrazioni=('registrazione', 'nunique'),
                                            cuscinetti=('cuscinetto', 'nunique'))
conteggi.index = [config.NOMI_CLASSI[c] for c in conteggi.index]
print(conteggi.to_string())
print()

corte = inv[inv['campioni'] < 4 * config.FS_ATTESO]
print('registrazioni con meno di 256000 campioni:', len(corte),
      '-> segmenti persi:', 4 * len(inv) - len(segmenti))
print('il paper ne dichiara', config.PAPER_SEGMENTI_DICHIARATI,
      '| differenza:', len(segmenti) - config.PAPER_SEGMENTI_DICHIARATI)

## 5. Il segnale, prima di darlo in pasto alla reteDue grandezze servono piu avanti: il valore efficace, perche il residuo dell'arco SELUgli e legato, e il picco, perche determina quanti campioni finiscono sotto il limiteinferiore della SELU.

In [ ]:
descrittive = pd.DataFrame([f.caratteristiche(x) for x in segmenti])
descrittive['classe'] = anagrafica['classe'].values
descrittive['cuscinetto'] = anagrafica['cuscinetto'].values

per_classe = descrittive.groupby('classe')[['rms', 'picco', 'std', 'crest', 'f_dominante']].mean()
per_classe.index = [config.NOMI_CLASSI[c] for c in per_classe.index]
print('media per classe:')
print(per_classe.to_string())
print()
print('media per cuscinetto:')
print(descrittive.groupby('cuscinetto')[['rms', 'picco', 'crest']].mean().to_string())
print()
print('valori non finiti:', int(descrittive['non_finiti'].sum()))

## 6. Frame e limite inferiore della SELULa segmentazione fine segue le equazioni 7-9 del paper: un frame contiene un giromeccanico completo. A 1500 rpm e 64 kHz sono 2560 campioni, quindi 25 frame persegmento.La SELU e limitata inferiormente da $-\lambda\alpha \approx -1{,}7581$: il decoder nonpuo produrre in uscita valori piu bassi, qualunque cosa impari. Con corrente nonnormalizzata e picchi intorno a 3 A, una parte dei campioni cade sotto quella sogliaed e irriproducibile per costruzione. Qui si misura quanti sono.Il paper non dice se il segnale venga normalizzato prima di entrare nel DAE. Nonessendoci alcun riferimento a una normalizzazione, qui si usa il segnale come e.

In [ ]:
lunghezza_frame = f.lunghezza_frame(config.REGIMI[regime]['rpm'])
frame_per_segmento = segmenti.shape[1] // lunghezza_frame
frame = segmenti.reshape(-1, lunghezza_frame)
classe_del_frame = np.repeat(anagrafica['classe'].values, frame_per_segmento)
segmento_del_frame = np.repeat(np.arange(len(segmenti)), frame_per_segmento)

print('un giro a', config.REGIMI[regime]['rpm'], 'rpm ->', lunghezza_frame, 'campioni')
print(len(frame), 'frame,', frame_per_segmento, 'per segmento')

# il reshape deve conservare l'ordine: il frame j del segmento i sta nella riga i*25+j
prova = np.array_equal(frame[3 * frame_per_segmento + 7],
                       segmenti[3, 7 * lunghezza_frame:8 * lunghezza_frame])
print('il reshape conserva l ordine dei frame:', prova)
print('il reshape inverso restituisce i segmenti:',
      np.array_equal(frame.reshape(len(segmenti), -1), segmenti))
print()

sotto = frame < config.PAVIMENTO_SELU
print('pavimento della SELU:', round(config.PAVIMENTO_SELU, 4))
print('campioni sotto il pavimento:', round(100 * float(np.mean(sotto)), 2), '%')
print('frame che ne contengono almeno uno:',
      round(100 * float(np.mean(sotto.any(axis=1))), 2), '%')

## 7. Le due architettureLe Tabelle 3 e 4 del paper riportano il numero di parametri di ogni strato. Sonosufficienti a ricostruire l'architettura senza ambiguita, e il conteggio totale serveda verifica: se coincide, la ricostruzione e corretta.Due dettagli non sono dichiarati dal paper e vanno ricavati o scelti. Il nucleo delleconvoluzioni si ricava: con un canale in ingresso e 64 filtri con bias, solo un nucleoda 3 da i 256 parametri del primo strato e i 6176 del secondo. L'inizializzazione deipesi va scelta, e qui e LeCun normale, l'unica per cui vale l'auto-normalizzazionedella SELU dimostrata da Klambauer et al.

In [ ]:
dae_prova = f.crea_dae()
cnn_prova = f.crea_cnn(segmenti.shape[1])

print('DAE:', f.conta_parametri(dae_prova), 'parametri')
print('CNN:', f.conta_parametri(cnn_prova), 'parametri')
print()

for nucleo in [2, 3, 4, 5]:
    primo = 64 * (nucleo * 1 + 1)
    secondo = 32 * (nucleo * 64 + 1)
    nota = '  <- coincide con la Tabella 4' if (primo, secondo) == (256, 6176) else ''
    print('nucleo', nucleo, '-> primo strato', primo, 'parametri, secondo', secondo, nota)

del dae_prova, cnn_prova

## 8. I frame sani per addestrare il DAEIl paper addestra il DAE su 2560 frame di cuscinetto sano, divisi 80:20 in 2048 perl'addestramento e 512 per il calcolo dell'errore di validazione. Non dice come venganoscelti fra i 12 000 disponibili, ne se la divisione sia casuale.Qui i 2560 sono estratti a sorte fra tutti i frame sani e poi **mescolati** prima delladivisione. Il mescolamento non e un dettaglio: senza di esso i primi 2048 indicicadrebbero tutti nei primi cuscinetti e la validazione finirebbe per essere compostaquasi interamente da K006, con un valore efficace sistematicamente piu basso dellamedia. L'errore di validazione risulterebbe piu basso di quello di addestramento, esarebbe un artefatto della divisione, non una proprieta del modello.

In [ ]:
frame_dae = 2560
frame_dae_addestramento = 2048
seme = 0

indici_sani = np.flatnonzero(classe_del_frame == 0)
rng = np.random.default_rng(seme)
scelti = rng.choice(indici_sani, size=frame_dae, replace=False)
rng.shuffle(scelti)

indici_dae_train = scelti[:frame_dae_addestramento]
indici_dae_val = scelti[frame_dae_addestramento:]

visto_dal_dae = np.zeros(len(frame), dtype=bool)
visto_dal_dae[scelti] = True

print('frame sani disponibili:', len(indici_sani),
      '| usati per il DAE:', frame_dae,
      '->', round(100 * frame_dae / len(indici_sani), 1), '%')
print('addestramento', len(indici_dae_train), '| validazione', len(indici_dae_val))
print()

cuscinetto_del_frame = np.repeat(anagrafica['cuscinetto'].values, frame_per_segmento)
composizione = pd.DataFrame({
    'addestramento': pd.Series(cuscinetto_del_frame[indici_dae_train]).value_counts(),
    'validazione': pd.Series(cuscinetto_del_frame[indici_dae_val]).value_counts(),
}).fillna(0).astype(int).sort_index()
print('da quali cuscinetti provengono i frame:')
print(composizione.to_string())

## 9. Addestramento dei due autoencoderIl paper prescrive l'attivazione SELU su tutti gli strati, uscita compresa (Tabella 3).Per stabilire quanta parte del residuo dipenda dal limite inferiore della SELU e quantadalla modellazione del sistema si addestra un secondo autoencoder identico in tuttotranne l'ultima attivazione, che e lineare. Stessi dati, stesso seme, stesse epoche:l'unica differenza e quella.500 epoche fisse, senza arresto anticipato, come dichiarato. Il passo di apprendimento(0,0003) e la dimensione del lotto (256) non sono dichiarati per il DAE: il lotto da 64che il paper indica si riferisce esplicitamente alla CNN.

In [ ]:
frame_train = frame[indici_dae_train]
frame_val = frame[indici_dae_val]

dae = {}
curve = {}
for nome, uscita_selu in [('selu', True), ('lineare', False)]:
    print('DAE con uscita', nome)
    modello, c_train, c_val = f.addestra_dae(frame_train, frame_val,
                                             uscita_selu=uscita_selu,
                                             epoche=500, lotto=256, passo=3e-4,
                                             seme=seme, dev=dev, stampa_ogni=50)
    dae[nome] = modello
    curve[nome] = {'addestramento': c_train, 'validazione': c_val}
    print()

## 10. Il residuoIl residuo e la differenza campione per campione fra il segnale e la sua ricostruzione(equazione 15). Il paper riporta un residuo medio di 0,104 per lo stato normale, 0,386per la pista esterna e 0,479 per la interna: rapporti di 3,71 e 4,61 rispetto al sano,ed e questa separazione a rendere il problema facile per la CNN a valle.La Sezione 3.4 del paper precisa che le istanze usate per generare il residuo sonodiverse da quelle di addestramento. Qui il residuo viene calcolato su tutti i frame,perche i segmenti servono interi alla CNN, ma il residuo medio della classe normaleviene riportato in entrambi i modi: su tutti i frame sani e sui soli frame che il DAEnon ha mai visto. La differenza fra i due valori misura l'entita esatta di questoscostamento, invece di lasciarla all'argomentazione.

In [ ]:
residui = {}
mse_frame = {}
for nome in dae:
    residui[nome] = f.calcola_residui(dae[nome], frame, dev=dev)
    mse_frame[nome] = f.mse_per_frame(residui[nome])

righe = []
for nome in dae:
    for classe, etichetta in enumerate(config.NOMI_CLASSI):
        del_classe = classe_del_frame == classe
        righe.append({
            'uscita': nome,
            'classe': etichetta,
            'residuo_tutti': float(np.mean(mse_frame[nome][del_classe])),
            'residuo_mai_visti': float(np.mean(mse_frame[nome][del_classe & ~visto_dal_dae])),
            'paper': config.PAPER_RESIDUO[etichetta],
        })

tabella_residui = pd.DataFrame(righe)
for nome in dae:
    parte = tabella_residui['uscita'] == nome
    base_tutti = tabella_residui.loc[parte, 'residuo_tutti'].iloc[0]
    base_visti = tabella_residui.loc[parte, 'residuo_mai_visti'].iloc[0]
    tabella_residui.loc[parte, 'rapporto'] = tabella_residui.loc[parte, 'residuo_tutti'] / base_tutti
    tabella_residui.loc[parte, 'rapporto_mai_visti'] = tabella_residui.loc[parte, 'residuo_mai_visti'] / base_visti
tabella_residui['rapporto_paper'] = tabella_residui['paper'] / config.PAPER_RESIDUO['normale']

print(tabella_residui.to_string(index=False))
print()

scarto = 100 * abs(tabella_residui['residuo_mai_visti'] - tabella_residui['residuo_tutti']) / tabella_residui['residuo_tutti']
print('scarto massimo fra i due modi di calcolare il residuo:', round(float(scarto.max()), 2), '%')

### Da dove viene il residuo dell'arco SELUDue misure dicono se il residuo porti informazione sul cuscinetto oppure sull'ampiezzadel segnale: la quota di energia del residuo che si concentra sui campioni sotto ilpavimento della SELU, e la correlazione fra il valore efficace di un segmento e il suoresiduo. L'arco lineare fa da termine di paragone.

In [ ]:
for nome in dae:
    r = residui[nome].astype(np.float64) ** 2
    quota = 100 * float(np.sum(r[sotto]) / np.sum(r))
    mse_segmento = mse_frame[nome].reshape(len(segmenti), frame_per_segmento).mean(axis=1)
    correlazione = float(np.corrcoef(descrittive['rms'].values, mse_segmento)[0, 1])
    print('uscita', nome)
    print('   energia del residuo che sta sui campioni sotto il pavimento:',
          round(quota, 2), '%')
    print('   correlazione fra valore efficace del segmento e suo residuo:',
          round(correlazione, 3))

## 11. Suddivisione e classificazioneIl paper suddivide 80/20 i campioni di residuo, senza vincoli sulla provenienza. Poicheogni registrazione da 4 s produce quattro segmenti consecutivi, segmenti della stessaregistrazione finiscono sia in addestramento sia in verifica. E una fuga di informazione,e viene riprodotta deliberatamente: correggerla qui renderebbe impossibile capire se loscarto rispetto al paper dipenda da questa scelta o da altro. Il confronto consuddivisioni piu severe e nel notebook dell'indagine.La stessa suddivisione viene usata da tutti i metodi confrontati, cosi il confrontoresta controllato.

In [ ]:
etichette = anagrafica['classe'].values.astype(np.int64)
indici_train, indici_test = f.suddividi(anagrafica, livello='segmento',
                                        frazione_test=0.2, seme=seme)

print(f.sovrapposizione(anagrafica, indici_train, indici_test))
print()
print('composizione della verifica:')
print(pd.Series(etichette[indici_test]).map(dict(enumerate(config.NOMI_CLASSI)))
      .value_counts().to_string())
print()
print('accuratezza della classe piu numerosa:',
      round(100 * f.baseline_degenere(etichette[indici_test])['accuratezza'], 2), '%')

In [ ]:
residui_segmento = {nome: residui[nome].reshape(len(segmenti), -1) for nome in residui}

ingressi = [('DAE + residuo + CNN', residui_segmento['selu']),
            ('DAE + residuo + CNN, uscita lineare', residui_segmento['lineare']),
            ('segnale grezzo + CNN', segmenti)]

risultati = {}
previsioni = {}
for etichetta, dati in ingressi:
    _, previste, vere, _ = f.addestra_cnn(dati, etichette, indici_train, indici_test,
                                          epoche=500, lotto=64, passo=3e-4,
                                          seme=seme, dev=dev, stampa_ogni=100,
                                          etichetta=etichetta)
    m = f.metriche(vere, previste)
    risultati[etichetta] = 100 * m['accuratezza']
    previsioni[etichetta] = (vere, previste)
    print(m['report'])
    print(pd.DataFrame(m['confusione'], index=config.NOMI_CLASSI,
                       columns=config.NOMI_CLASSI).to_string())
    print()

## 12. Ablazione con le caratteristiche statisticheLa Tabella 5 del paper elenca dieci grandezze estratte dal residuo, date poi in pasto auna macchina a vettori di supporto, a una foresta casuale e a un k-nearest neighbor.Sono tutte e dieci: valore efficace, energia, deviazione standard, curtosi, varianza,asimmetria, fattore di cresta, entropia di Shannon, fattore di forma ed entropialog-energetica.Vale la pena guardare quanto siano davvero indipendenti fra loro, perche quattro di essesono funzioni monotone della stessa quantita.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

caratteristiche = f.feature_tabella5(residui_segmento['selu'])
correlazioni = pd.DataFrame(np.corrcoef(caratteristiche, rowvar=False),
                            index=f.NOMI_FEATURE_TABELLA5,
                            columns=f.NOMI_FEATURE_TABELLA5)
print('correlazione fra le dieci caratteristiche (valore assoluto sopra 0,99):')
alta = correlazioni.abs().where(np.triu(np.ones(correlazioni.shape), k=1).astype(bool)).stack()
print(alta[alta > 0.99].round(4).to_string())
print()

for nome in residui_segmento:
    caratteristiche = f.feature_tabella5(residui_segmento[nome])
    scalatore = StandardScaler().fit(caratteristiche[indici_train])
    X_train = scalatore.transform(caratteristiche[indici_train])
    X_test = scalatore.transform(caratteristiche[indici_test])

    classificatori = {'SVM': SVC(random_state=seme),
                      'RF': RandomForestClassifier(n_estimators=200, random_state=seme),
                      'KNN': KNeighborsClassifier()}
    print('caratteristiche del residuo, uscita', nome)
    for sigla, classificatore in classificatori.items():
        classificatore.fit(X_train, etichette[indici_train])
        accuratezza = 100 * float(np.mean(classificatore.predict(X_test) == etichette[indici_test]))
        print('   residuo + caratteristiche +', sigla, '->', round(accuratezza, 2), '%')
        if nome == 'selu':
            risultati['residuo + feature + ' + sigla] = accuratezza
    print()

## 13. Riepilogo, figure e salvataggio

In [ ]:
confronto = pd.DataFrame({'metodo': list(risultati), 'nostro': list(risultati.values())})
confronto['paper'] = confronto['metodo'].map(config.PAPER_ACCURATEZZA)
confronto['scarto'] = confronto['nostro'] - confronto['paper']
print(confronto.to_string(index=False))

In [ ]:
fig, assi = plt.subplots(1, 3, figsize=(13, 3.6))

for nome in curve:
    assi[0].plot(curve[nome]['addestramento'], label='addestramento, ' + nome, lw=1)
    assi[0].plot(curve[nome]['validazione'], label='validazione, ' + nome, lw=1, ls='--')
assi[0].set_yscale('log')
assi[0].set_xlabel('epoca'); assi[0].set_ylabel('errore quadratico medio')
assi[0].set_title('Addestramento dei due autoencoder')
assi[0].legend(fontsize=7)

larghezza = 0.38
posizioni = np.arange(3)
for spostamento, (nome, colore) in zip([-larghezza / 2, larghezza / 2],
                                       [('selu', f.COLORI['nostro']),
                                        ('lineare', f.COLORI['normale'])]):
    parte = tabella_residui[tabella_residui['uscita'] == nome]
    assi[1].bar(posizioni + spostamento, parte['rapporto'], larghezza,
                label='uscita ' + nome, color=colore)
assi[1].plot(posizioni, tabella_residui[tabella_residui['uscita'] == 'selu']['rapporto_paper'],
             'o--', color=f.COLORI['accento'], label='paper')
assi[1].set_xticks(posizioni); assi[1].set_xticklabels(config.NOMI_CLASSI)
assi[1].set_ylabel('residuo rapportato al sano')
assi[1].set_title('Separazione delle classi nel residuo')
assi[1].legend(fontsize=7)

ordine = confronto.sort_values('nostro')
posizioni = np.arange(len(ordine))
assi[2].barh(posizioni - 0.2, ordine['nostro'], 0.4, label='replica', color=f.COLORI['nostro'])
assi[2].barh(posizioni + 0.2, ordine['paper'], 0.4, label='paper', color=f.COLORI['normale'])
assi[2].set_yticks(posizioni)
assi[2].set_yticklabels([m.replace(', uscita', '\n uscita') for m in ordine['metodo']], fontsize=7)
assi[2].set_xlabel('accuratezza [%]')
assi[2].set_title('Confronto con i valori dichiarati')
assi[2].legend(fontsize=7)

f.salva_figura(fig, 'replica_riepilogo', P['figure'])
plt.show()

In [ ]:
f.salva_tabella(confronto, 'confronto_accuratezza', P['tabelle'])
f.salva_tabella(tabella_residui, 'confronto_residui', P['tabelle'])
f.salva_tabella(riepilogo, 'inventario_regimi', P['tabelle'])
f.salva_tabella(anagrafica, 'anagrafica_segmenti', P['tabelle'])
f.salva_tabella(descrittive.groupby('cuscinetto')[['rms', 'picco', 'crest']].mean().reset_index(),
                'statistiche_per_cuscinetto', P['tabelle'])

import torch
for nome in dae:
    percorso = os.path.join(P['risultati'], 'dae_uscita_' + nome + '.pt')
    torch.save(dae[nome].state_dict(), percorso)
    print('modello salvato:', percorso)

print()
for cartella in [P['figure'], P['tabelle'], P['risultati']]:
    for nome in sorted(os.listdir(cartella)):
        percorso = os.path.join(cartella, nome)
        if os.path.isfile(percorso):
            print(nome, round(os.path.getsize(percorso) / 1e6, 2), 'MB')